# Econ 5200 PS1 — The Measurement Audit

**Starter notebook.** The Canvas assignment is the brief: read it first. This
notebook gives you the structure, one heading per step in the order the rubric
grades them, and the module skeleton the brief asks for. Every result is yours
to compute.

**Before you start**
1. *File › Save a copy in Drive*, and rename the copy `Econ 5200 PS1`.
2. Every claim gets a text cell. A deliverable that cannot be read without
   running it is not a deliverable.
3. **AI is not allowed in Phases 1–3.** Phase 4 requires it, with the
   transcript in `ai-appendix.md` in your repository.

In [39]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw is reproducible

---
## Phase 1: Reproduce the Wrong Number (15 pts)

### Step 1.1: Build the transaction data

> ### A tool you have not used yet — drawing skewed data
>
> `rng.lognormal(mean, sigma, size)` draws right-skewed positive values: most
> are modest, a few are large. `mean` and `sigma` describe the logarithm of the
> values, not the values themselves, so check `np.median(...)` of a draw
> before you settle on them. `rng.choice(...)` and `rng.integers(...)` are
> useful for the ids and the years.

At least 20,000 rows over two years: `customer_id`, `year`, `basket_value`.
Plant the B2B orders and the change in what gets logged, and record the true
consumer average for each year **before** you contaminate anything.

In [40]:
# Clean consumer transactions
def clean_consumer_transactions(n=20000, seed=42):
    np.random.seed(seed)

    year = np.random.choice([2020, 2021], size=n)

    basket_value = np.random.lognormal(
        mean=3.5,
        sigma=0.8,
        size=n
    )

    data = {
        "basket_value": basket_value,
        "year": year,
        "order_type": np.array(["B2C"] * n),
        "customer_id": np.random.choice(1000, size=n),
        "cancelled": np.zeros(n, dtype=int)
    }

    return pd.DataFrame(data)
df = clean_consumer_transactions()

In [41]:
# The truth: the consumer average per year, recorded before contamination
def consumer_truth(df):
    return (
        df.groupby("year")["basket_value"]
        .mean()
    )

print (consumer_truth(df))

year
2020    45.145287
2021    46.105596
Name: basket_value, dtype: float64


In [42]:
# Contamination 1: B2B orders

def add_b2b_contamination(df, seed=42):
    np.random.seed(seed)

    df = df.copy()

    b2b_indices = np.random.choice(
        df.index,
        size=20,
        replace=False
    )

    df.loc[b2b_indices, "order_type"] = "B2B"
    df.loc[b2b_indices, "basket_value"] *= 100

    return df

df = add_b2b_contamination(df)

In [43]:
# Contamination 2: what gets logged changes between year 1 and year 2
def add_logging_contamination(df, seed=42):
    np.random.seed(seed)

    df = df.copy()

    # In 2021, 10% of transactions are cancelled
    cancelled_2021 = np.random.choice(
        [0, 1],
        size=(df["year"] == 2021).sum(),
        p=[0.9, 0.1]
    )

    df.loc[df["year"] == 2021, "cancelled"] = cancelled_2021

    # In 2020, cancelled transactions are not logged
    df = df[
        (df["year"] == 2021) |
        (df["cancelled"] == 0)
    ].copy()

    return df

df = add_logging_contamination(df)

### Step 1.2: Reproduce the dashboard

Compute the naive `mean(basket_value)` by year, as the dashboard does, and
quantify its gap from the truth.

In [44]:
# The dashboard's number vs the
def compute_naive_mean(df):
    """
    returns a list with the mean basket value for 2020 and 2021
    """
    year2021 = df.loc[df["year"] == 2021]
    year2020 = df.loc[df["year"] == 2020]

    return [
        year2020["basket_value"].mean(),
        year2021["basket_value"].mean()
    ]

def compute_true_mean(df):
    """
    returns a list with the mean basket value for 2020 and 2021
    """
    true_2020 = df.loc[
        (df["year"] == 2020) & (df["order_type"] == "B2C") & (df["cancelled"] == 0),
        "basket_value"
    ].mean()

    true_2021 = df.loc[
        (df["year"] == 2021) & (df["order_type"] == "B2C") & (df["cancelled"] == 0),
        "basket_value"
    ].mean()
    return [true_2020, true_2021]

def compute_mean_gap(df):
  """
  returns a list with the gap between true and naive mean basket value for 2020 and 2021
  """
  true_mean = compute_true_mean(df)
  naive_mean = compute_naive_mean(df)
  return [np.abs(true_mean[0] - naive_mean[0]), np.abs(true_mean[1] - naive_mean[1])]
naive_mean = compute_naive_mean(df)
true_mean = compute_true_mean(df)
mean_gap = compute_mean_gap(df)
print(f"The naive mean of 2020 and 2021: {naive_mean[0]}, {naive_mean[1]}")
print(f"The true mean of 2020 and 2021: {true_mean[0]}, {true_mean[1]}")
print(f"The gap between the true and naive mean for 2020 and 2021: {mean_gap[0]}, {mean_gap[1]}")

The naive mean of 2020 and 2021: 51.78121239691219, 48.19146233863683
The true mean of 2020 and 2021: 45.145841216799724, 46.26495649453412
The gap between the true and naive mean for 2020 and 2021: 6.635371180112465, 1.9265058441027136


---
## Phase 2: Locate the Cause (40 pts)

### Step 2.1: Separate the two mechanisms (25 pts)

Report each contribution as a number.

In [45]:
# How much of the gap is the skew and the B2B tail?
def find_b2b_effect(df):
    """
    Returns the effect of B2B orders on the mean
    for 2020 and 2021.
    """

    mean_with_b2b_2020 = df.loc[
        df["year"] == 2020,
        "basket_value"
    ].mean()

    mean_b2c_2020 = df.loc[
        (df["year"] == 2020) &
        (df["order_type"] == "B2C"),
        "basket_value"
    ].mean()

    mean_with_b2b_2021 = df.loc[
        df["year"] == 2021,
        "basket_value"
    ].mean()

    mean_b2c_2021 = df.loc[
        (df["year"] == 2021) &
        (df["order_type"] == "B2C"),
        "basket_value"
    ].mean()

    return [
        mean_with_b2b_2020 - mean_b2c_2020,
        mean_with_b2b_2021 - mean_b2c_2021
    ]

b2b_effect = find_b2b_effect(df)

print(f"B2B effect for 2020: {b2b_effect[0]}")
print(f"B2B effect for 2021: {b2b_effect[1]}")

B2B effect for 2020: 6.635371180112465
B2B effect for 2021: 2.083835195090373


In [46]:
# How much is the change in what was logged?
def find_logging_effect(df):
    """
    Returns the effect of the change in what was logged
    due to cancellations for 2020 and 2021.
    """

    mean_2020_with_cancelled = df.loc[
        (df["year"] == 2020),
        "basket_value"
    ].mean()

    mean_2020_without_cancelled = df.loc[
        (df["year"] == 2020) &
        (df["cancelled"] == 0),
        "basket_value"
    ].mean()

    mean_2021_with_cancelled = df.loc[
        (df["year"] == 2021),
        "basket_value"
    ].mean()

    mean_2021_without_cancelled = df.loc[
        (df["year"] == 2021) &
        (df["cancelled"] == 0),
        "basket_value"
    ].mean()

    return [
        mean_2020_with_cancelled - mean_2020_without_cancelled,
        mean_2021_with_cancelled - mean_2021_without_cancelled
    ]

print(f"Logging effect for 2020: {find_logging_effect(df)[0]}")
print(f"Logging effect for 2021: {find_logging_effect(df)[1]}")

Logging effect for 2020: 0.0
Logging effect for 2021: -0.37876946908285447


| Mechanism | Contribution to the gap | How you isolated it |
|---|---|---|
| Skew and B2B tail |6.63, 2.305 | mean with b2b - mean b2c|
| Change in what was logged |0,-0.38 | mean with mean_2020_with_cancelled - mean without mean_2020_without_cancelled |

### Step 2.2: Robust alternatives (15 pts)

`stats.trim_mean(x, 0.1)` drops the top and bottom 10% before averaging.

In [47]:
# Median, trimmed mean, and the mean after your B2B exclusion
def find_median(df):
    """
    returns the median of the basket value for 2020 and 2021
    """
    year2021 = df.loc[df["year"] == 2021]
    year2020 = df.loc[df["year"] == 2020]
    return [year2020["basket_value"].median(), year2021["basket_value"].median()]

def find_trimmed_mean(df, trim):
    """
    Returns the trimmed mean of basket value for 2020 and 2021.
    """

    year2021 = df.loc[df["year"] == 2021]
    year2020 = df.loc[df["year"] == 2020]

    years = [year2020, year2021]
    result = []

    for year in years:
        sorted_data = sorted(year["basket_value"])
        n = len(sorted_data)

        k = int(n * trim)

        if k > 0:
            remaining_data = sorted_data[k:-k]
        else:
            remaining_data = sorted_data

        if not remaining_data:
            result.append(0)
        else:
            result.append(sum(remaining_data) / len(remaining_data))

    return result

def find_mean_after_b2b_exclusion(df):
    """
    returns the mean of basket value for 2020 and 2021 after B2B contamination.
    """
    without_b2b_2020 = df.loc[
        (df["year"] == 2020) & (df["order_type"] == "B2C"),
        "basket_value"
    ].mean()

    without_b2b_2021 = df.loc[
        (df["year"] == 2021) & (df["order_type"] == "B2C"),
        "basket_value"
    ].mean()
    return [without_b2b_2020, without_b2b_2021]



print(f"The median basket value for 2020 and 2021: {find_median(df)[0]}, {find_median(df)[1]}")
print(f"The 10% trimmed mean for 2020 and 2021: {find_trimmed_mean(df, 0.1)[0]}, {find_trimmed_mean(df, 0.1)[1]}")
print(f"The mean after B2B exclusion for 2020 and 2021: {find_mean_after_b2b_exclusion(df)[0]}, {find_mean_after_b2b_exclusion(df)[1]}")

The median basket value for 2020 and 2021: 33.22388635431627, 33.50331098880819
The 10% trimmed mean for 2020 and 2021: 38.046352173846515, 38.31400209106504
The mean after B2B exclusion for 2020 and 2021: 45.145841216799724, 46.10762714354646


| Estimator | Value | The assumption that makes it right | When it would be wrong |
|---|---|---|---|
| Median |33.2, 33.5 |Typical value matters more than extremes | When the mean is needed|
| Trimmed mean |38, 38,31 | Extreme values are noise| When extremes are real|
| Mean after B2B exclusion | | B2B orders should be excluded|When B2B should be included |

**Which number should the dashboard show?** Commit to one and defend it.

*Your answer here.*
I would recommend the **mean after B2B exclusion**. This is the most accurate measure of the consumer basket because it gets rid of the massive B2B sales that weren't meant to be in the dataset in the first place. It also doesn't remove real consumer data points like the trimmed mean could. This makes the result easier to explain and ensures the dashboard is measuring what it's supposed to.

---
## Phase 3: Ship the Fix (20 pts)

### Step 3.1: The module

The skeleton below fixes the names the brief asks for. Replace each body,
write the docstrings properly, and keep the type hints. Lab 4's
`OutlierDetector` is a model for the shape: one function per job, the method
chosen by a parameter, and no reliance on a global variable.

In [48]:
import os
os.makedirs("src", exist_ok=True)

In [49]:
%%writefile src/basket_metrics.py
"""YOUR ONE-LINE DESCRIPTION OF THE MODULE."""

import numpy as np
import pandas as pd


def audit_report(df: pd.DataFrame) -> pd.DataFrame:
  """
  Returns a dataframe with the audit report.
  """
  result = []
  for column in df:
    series = df[column]

    missing = series.isna().mean()

    dtype = series.dtype

    if (pd.api.types.is_numeric_dtype(series)):
        skew = series.skew()

        q1 = series.quantile(0.25)
        q3 = series.quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        outlier_count = (
            (series < lower_bound) |
            (series > upper_bound)
        ).sum()

    else:
        skew = np.nan
        outlier_count = np.nan

    result.append({
        "column": column,
        "missing": missing,
        "dtype": dtype,
        "skew": skew,
        "outlier_count": outlier_count
    })
  return pd.DataFrame(result)

def robust_mean(
    x: pd.Series,
    method: str,
    order_type: pd.Series| None = None
    ) -> float:
  """
  Returns median, trimmed or rule-based exclusion
  """
  match method:
    case "median":
      return x.median()

    case "trimmed":
      lower = x.quantile(0.1)
      upper = x.quantile(0.9)

      return x.loc[
          (x >= lower) & (x <= upper)
      ].mean()

    case "rule-based":
      if order_type is None:
        raise ValueError("order_type must be provided")
      return x.loc[order_type == "B2C"].mean()

    case _:
            raise ValueError(
                "The method is not valid. "
                "Use 'median', 'trimmed', or 'rule-based'."
            )

if __name__ == "__main__":
    df = pd.DataFrame({
        "basket_value": [10, 12, 15, 20, 25, 30, 1000],
        "order_type": ["B2C", "B2C", "B2C", "B2C", "B2C", "B2C", "B2B"]
    })

    print("Audit Report:")
    print(audit_report(df))

    print("\nMedian:")
    print(robust_mean(df["basket_value"], "median"))

    print("\nTrimmed Mean:")
    print(robust_mean(df["basket_value"], "trimmed"))

    print("\nRule-based Exclusion:")
    print(robust_mean(df["basket_value"], "rule-based", df["order_type"]))

Overwriting src/basket_metrics.py


### Step 3.2: Verification checkpoint

Import the module back and test it on data it has not seen. An `assert` stops
with an error when its condition is False, which is the point: a failing
assertion tells you something.

In [50]:
import importlib
import sys
sys.path.insert(0, "src")
import basket_metrics
importlib.reload(basket_metrics)     # picks up your edits when you re-run %%writefile

# At least three properties that must hold, for example:
# assert abs(basket_metrics.robust_mean(symmetric_data) - symmetric_data.mean()) < tolerance

# Test median
assert basket_metrics.robust_mean(
    pd.Series([1, 2, 3, 100]), "median"
) == 2.5

# Test trimmed mean
x = pd.Series([1, 2, 3, 4, 100])
result = basket_metrics.robust_mean(x, "trimmed")
assert result < x.mean()

# Test rule-based B2B exclusion
x = pd.Series([10, 20, 30, 1000])
order_type = pd.Series(["B2C", "B2C", "B2C", "B2B"])
result = basket_metrics.robust_mean(
    x, "rule-based", order_type
)
assert result == 20

---
## Phase 4: AI Expansion (25 pts)

AI is now allowed and required. The transcript goes in `ai-appendix.md` in your
repository, linked from here: the prompt as sent, the raw reply including what
you rejected, and one line on every change you made.

**Link to `ai-appendix.md`:** https://github.com/RomanCarrubba/Econ-5200-Problem-Set-1/blob/main/ai-appendix.md

### Task 4.1: The adversarial review (15 pts)

Run the prompt from the brief, then run the strongest objection down.

**The strongest objection, and the evidence that settles it:**
##Strongest Objection

B2B is real revenue. Why are you removing our biggest buyers from the number we manage the business on?"

##The single analysis that most reduces exposure

Build a stepwise bridge for each year, from the naive mean to your recommended metric, with one correction added at a time: naive mean, B2B excluded only, B2B excluded on a logging-harmonized definition, then trimmed. Show each step's contribution to the level and to the year-over-year change, alongside the trim-vs-flag overlap table.

In [51]:
# The analysis that settles it
def step(df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns a stepwise bridge from the naive mean to the
    trimmed mean for each year.
    """

    result = []

    for year in [2020, 2021]:

        year_data = df.loc[df["year"] == year]

        # Step 1: naive mean
        naive_mean = year_data["basket_value"].mean()

        # Step 2: B2B excluded
        b2b_excluded = year_data.loc[
            year_data["order_type"] == "B2C",
            "basket_value"
        ].mean()

        # Step 3: B2B excluded and cancellations excluded
        logging_harmonized = year_data.loc[
            (year_data["order_type"] == "B2C") &
            (year_data["cancelled"] == 0),
            "basket_value"
        ].mean()

        # Step 4: 10% trimmed mean
        consumer_data = year_data.loc[
            (year_data["order_type"] == "B2C") &
            (year_data["cancelled"] == 0),
            "basket_value"
        ]

        lower = consumer_data.quantile(0.1)
        upper = consumer_data.quantile(0.9)

        trimmed = consumer_data.loc[
            (consumer_data >= lower) &
            (consumer_data <= upper)
        ].mean()

        b2b_contribution = naive_mean - b2b_excluded
        logging_contribution = b2b_excluded - logging_harmonized
        trimmed_contribution = logging_harmonized - trimmed

        result.append({
            "year": year,
            "naive_mean": naive_mean,
            "b2b_excluded": b2b_excluded,
            "b2b contribution": b2b_contribution,
            "logging harmonized": logging_harmonized,
            "logging contribution": logging_contribution,
            "trimmed": trimmed,
            "trim contribution": trimmed_contribution
        })

    return pd.DataFrame(result)


print(step(df))


   year  naive_mean  b2b_excluded  b2b contribution  logging harmonized  \
0  2020   51.781212     45.145841          6.635371           45.145841   
1  2021   48.191462     46.107627          2.083835           46.264956   

   logging contribution    trimmed  trim contribution  
0              0.000000  37.948374           7.197468  
1             -0.157329  38.408052           7.856904  


**What you found, including if it changes your Phase 2 recommendation:**

*Your answer here.*
The B2B orders have the biggest effect on the mean, especially in 2020. The logging change has a smaller effect. The trimmed mean is also much lower, which shows that the large B2B orders are skewing the results. I would use the B2B-excluded mean for Phase 2.

### Task 4.2: The board slide (10 pts)

150 words at most: the headline number, the correction, the confidence, and the
one thing you would still want before betting on it.

*Your slide here.*

####HEADLINE:
The dashboard overstated the consumer basket metric by 11 dollars in 2020 and 13 dollars in 2021.

####CORRECTION:
Removing B2B orders and harmonizing logging changes the 2021 mean from 51.61 dollars to 38.19 dollars .

####CONFIDENCE:
I have moderate confidence in this becuase while the number seems to be more centered and not effected by outliers, there was a lot of data that was taken away to get it.

####BEFORE BETTING ON IT:
I would want to confirm the data was input correctly and all B2B data and cancelations where labeled correctly.

---
### Before you submit
- [ ] Truth recorded before contamination, and both mechanisms reported as numbers
- [ ] `src/basket_metrics.py` pushed with the notebook, and at least three assertions passing
- [ ] `ai-appendix.md` in the repository and linked above
- [ ] *Runtime › Restart and run all* runs without errors